
# BERT-Based Stance Classification with 10-fold Cross-Validation

This notebook trains a BERT-based text classifier to predict stance from the provided dataset using Stratified K-fold cross-validation (k=10).

## Notebook Overview
- Data preparation
- Tokenization and dataset preparation
- Model training using Hugging Face Transformers
- Evaluation and cross-validation


In [ ]:

# Install required libraries (uncomment if running for the first time)
# !pip install transformers torch sklearn numpy pandas


In [ ]:

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# Load dataset
data = pd.read_csv('stance_classification_temporal_dataset.csv')
texts = data['content'].fillna("").tolist()
labels = data['stance'].tolist()


In [ ]:

# Tokenizer setup
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Dataset class definition
class StanceDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:

kf = StratifiedKFold(n_splits=10)
fold_results = []

for fold, (train_index, test_index) in enumerate(kf.split(texts, labels)):
    print(f"Starting Fold {fold + 1}")
    
    # Data split
    X_train = [texts[i] for i in train_index]
    X_test = [texts[i] for i in test_index]
    y_train = [labels[i] for i in train_index]
    y_test = [labels[i] for i in test_index]

    # Dataset creation
    train_dataset = StanceDataset(X_train, y_train)
    test_dataset = StanceDataset(X_test, y_test)

    # Model initialization
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

    # Training arguments
    training_args = TrainingArguments(
        output_dir='./results', num_train_epochs=1, per_device_train_batch_size=8,
        per_device_eval_batch_size=8, evaluation_strategy='no', logging_steps=10,
        save_steps=10, logging_dir='./logs', seed=42
    )

    # Trainer setup
    trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset)

    # Model training
    trainer.train()

    # Prediction and evaluation
    predictions = trainer.predict(test_dataset)
    preds = np.argmax(predictions.predictions, axis=-1)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')
    report = classification_report(y_test, preds, output_dict=True)

    fold_results.append({
        'fold': fold + 1,
        'accuracy': acc,
        'f1_score': f1,
        'classification_report': report
    })

# Average metrics
avg_accuracy = np.mean([result['accuracy'] for result in fold_results])
avg_f1_score = np.mean([result['f1_score'] for result in fold_results])

print(f'Average Accuracy: {avg_accuracy:.2f}, Average F1 Score: {avg_f1_score:.2f}')

# Detailed Results DataFrame
results_df = pd.DataFrame(fold_results)
results_df
